In [1]:
%pip install polars scikit-learn matplotlib seaborn scipy umap-learn


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import polars as pl
import seaborn as sns
import os

import matplotlib.pyplot as plt

auth_path = "./data/cyber1/auth.txt"
redteam_path = "./data/cyber1/redteam.txt"

# Detect column count from a small sample
sample = pl.read_csv(auth_path, has_header=False, n_rows=5)
default_cols = [
    "time", "src_user", "dst_user", "src_pc", "dst_pc",
    "auth_type", "logon_type", "auth_orient", "success"
]
col_names = default_cols if sample.width == len(default_cols) else [f"col_{i}" for i in range(sample.width)]

# Lazy scan for large-file processing
auth_lf = pl.scan_csv(
    auth_path,
    has_header=False,
    new_columns=col_names,
    infer_schema_length=10_000,
    ignore_errors=True
)

# Basic shape and null counts
# row_count = auth_lf.select(pl.len().alias("rows")).collect(engine="streaming")
# null_counts = auth_lf.select([pl.col(c).null_count().alias(c) for c in col_names]).collect(engine="streaming")

# print("Rows:")
# print(row_count)
print("\nColumns:")
print(col_names)
# print("\nNull counts:")
# print(null_counts)

# Quick sample
print("\nSample rows:")
print(auth_lf.head(5).collect(engine="streaming"))

# Detect schema width and set column names
red_sample = pl.read_csv(redteam_path, has_header=False, n_rows=5)
red_default_cols = ["time", "user", "src_pc", "dst_pc"]
red_col_names = red_default_cols if red_sample.width == len(red_default_cols) else [f"col_{i}" for i in range(red_sample.width)]

red_lf = pl.scan_csv(
    redteam_path,
    has_header=False,
    new_columns=red_col_names,
    infer_schema_length=10_000,
    ignore_errors=True
)

# Basic overview
red_rows = red_lf.select(pl.len().alias("rows")).collect(engine="streaming")
red_nulls = red_lf.select([pl.col(c).null_count().alias(c) for c in red_col_names]).collect(engine="streaming")

print("Rows:")
print(red_rows)
print("\nColumns:")
print(red_col_names)
print("\nNull counts:")
print(red_nulls)
print("\nSample rows:")
print(red_lf.head(10).collect(engine="streaming"))



Columns:
['time', 'src_user', 'dst_user', 'src_pc', 'dst_pc', 'auth_type', 'logon_type', 'auth_orient', 'success']

Sample rows:
shape: (5, 9)
┌──────┬──────────────┬──────────────┬────────┬───┬───────────┬────────────┬─────────────┬─────────┐
│ time ┆ src_user     ┆ dst_user     ┆ src_pc ┆ … ┆ auth_type ┆ logon_type ┆ auth_orient ┆ success │
│ ---  ┆ ---          ┆ ---          ┆ ---    ┆   ┆ ---       ┆ ---        ┆ ---         ┆ ---     │
│ i64  ┆ str          ┆ str          ┆ str    ┆   ┆ str       ┆ str        ┆ str         ┆ str     │
╞══════╪══════════════╪══════════════╪════════╪═══╪═══════════╪════════════╪═════════════╪═════════╡
│ 1    ┆ ANONYMOUS    ┆ ANONYMOUS    ┆ C1250  ┆ … ┆ NTLM      ┆ Network    ┆ LogOn       ┆ Success │
│      ┆ LOGON@C586   ┆ LOGON@C586   ┆        ┆   ┆           ┆            ┆             ┆         │
│ 1    ┆ ANONYMOUS    ┆ ANONYMOUS    ┆ C586   ┆ … ┆ ?         ┆ Network    ┆ LogOff      ┆ Success │
│      ┆ LOGON@C586   ┆ LOGON@C586   ┆        ┆ 

In [3]:
# Label auth events using redteam events (1 = redteam/malicious, 0 = normal)

# Build redteam keys aligned to auth schema
red_keys = (
    red_lf.select(
        pl.col("time").cast(pl.Int64, strict=False).alias("time"),
        pl.col("user").alias("src_user"),
        pl.col("src_pc"),
        pl.col("dst_pc"),
    )
    .unique()
    .with_columns(pl.lit(1).alias("label"))
)

# Prepare auth and join labels
labeled_auth_lf = (
    auth_lf.with_columns(pl.col("time").cast(pl.Int64, strict=False).alias("time"))
    .join(red_keys, on=["time", "src_user", "src_pc", "dst_pc"], how="left")
    .with_columns(pl.col("label").fill_null(0).cast(pl.Int8))
)

# Quick check: class distribution
label_dist = (
    labeled_auth_lf.group_by("label")
    .agg(pl.len().alias("count"))
    .sort("label")
    .collect(engine="streaming")
)
print(label_dist)

# Optional sample of labeled rows
print(labeled_auth_lf.filter(pl.col("label") == 1).head(10).collect(engine="streaming"))

shape: (2, 2)
┌───────┬────────────┐
│ label ┆ count      │
│ ---   ┆ ---        │
│ i8    ┆ u32        │
╞═══════╪════════════╡
│ 0     ┆ 1051429757 │
│ 1     ┆ 702        │
└───────┴────────────┘
shape: (10, 10)
┌────────┬────────────┬────────────┬────────┬───┬────────────┬─────────────┬─────────┬───────┐
│ time   ┆ src_user   ┆ dst_user   ┆ src_pc ┆ … ┆ logon_type ┆ auth_orient ┆ success ┆ label │
│ ---    ┆ ---        ┆ ---        ┆ ---    ┆   ┆ ---        ┆ ---         ┆ ---     ┆ ---   │
│ i64    ┆ str        ┆ str        ┆ str    ┆   ┆ str        ┆ str         ┆ str     ┆ i8    │
╞════════╪════════════╪════════════╪════════╪═══╪════════════╪═════════════╪═════════╪═══════╡
│ 150885 ┆ U620@DOM1  ┆ U620@DOM1  ┆ C17693 ┆ … ┆ Network    ┆ LogOn       ┆ Success ┆ 1     │
│ 151036 ┆ U748@DOM1  ┆ U748@DOM1  ┆ C17693 ┆ … ┆ Network    ┆ LogOn       ┆ Success ┆ 1     │
│ 151648 ┆ U748@DOM1  ┆ U748@DOM1  ┆ C17693 ┆ … ┆ Network    ┆ LogOn       ┆ Success ┆ 1     │
│ 151993 ┆ U6115@DOM1 ┆ U6

In [4]:
# Keep all red-team events (label=1) and stratified-sample normal events (label=0)

# --- config ---
neg_to_pos_ratio = 20  # number of normal events per red-team event
strata_cols = ["auth_type", "logon_type", "auth_orient", "success"]
random_seed = 42

# Counts
# counts = (
#     labeled_auth_lf.group_by("label")
#     .agg(pl.len().alias("n"))
#     .collect()
# )

n_pos = label_dist['count'][1]
n_neg = label_dist['count'][0]
target_neg = min(n_neg, n_pos * neg_to_pos_ratio)

print(f"\nTarget negative samples: {target_neg} (out of {n_neg} available)")

# # positive samples
pos_lf = labeled_auth_lf.filter(pl.col("label") == 1)

# # negative samples
neg_lf = labeled_auth_lf.filter(pl.col("label") == 0)

neg_sampled_lf = neg_lf.gather_every(n_neg // target_neg)

final_sample_lf = pl.concat([pos_lf, neg_sampled_lf], how="vertical_relaxed")



# # Split
# pos_lf = labeled_auth_lf.filter(pl.col("label") == 1)
# neg_lf = labeled_auth_lf.filter(pl.col("label") == 0)

# if target_neg > 0 and n_neg > 0:
#     # Per-stratum allocation (proportional)
#     alloc_lf = (
#         neg_lf.group_by(strata_cols)
#         .agg(pl.len().alias("stratum_n"))
#         .with_columns(
#             (
#                 (pl.col("stratum_n") * (target_neg / n_neg))
#                 .round(0)
#                 .cast(pl.Int64)
#                 .clip(1, pl.col("stratum_n"))
#             ).alias("take_n")
#         )
#     )

#     # Deterministic pseudo-random order within each stratum, then keep first take_n rows
#     neg_sampled_lf = (
#         neg_lf.join(alloc_lf, on=strata_cols, how="inner")
#         .with_columns(
#             pl.struct(["time", "src_user", "dst_user", "src_pc", "dst_pc"])
#             .hash(seed=random_seed)
#             .alias("_rand")
#         )
#         .with_columns(
#             pl.col("_rand")
#             .rank(method="ordinal", descending=False)
#             .over(strata_cols)
#             .alias("_rnk")
#         )
#         .filter(pl.col("_rnk") <= pl.col("take_n"))
#         .drop(["_rand", "_rnk", "stratum_n", "take_n"])
#     )

#     final_sample_lf = pl.concat([pos_lf, neg_sampled_lf], how="vertical_relaxed")
# else:
#     final_sample_lf = pos_lf

# # Quick check
# final_dist = (
#     final_sample_lf.group_by("label")
#     .agg(pl.len().alias("count"))
#     .sort("label")
#     .collect(engine="streaming")
# )
# print(final_dist)

# # Materialize if needed
# final_sample_df = final_sample_lf.collect(engine="streaming")
# print(final_sample_df.head(10))


Target negative samples: 14040 (out of 1051429757 available)


In [5]:
# Optional: save to disk
final_sample_lf.collect(engine="streaming").write_csv("./data/cyber1/labeled_sample.csv")

In [7]:
skip_count = n_neg // target_neg
print(f"\nSampled every {skip_count}th normal event to achieve target ratio.")


Sampled every 74888th normal event to achieve target ratio.
